# Aula 6 — YOLOv8n EPI, 50 épocas

No Colab, selecione **Runtime > Change runtime type > T4 GPU** e execute **Runtime > Run all**. Ao final, este notebook gera `best.pt`, imprime mAP/precisão/recall, exporta uma detecção com bounding boxes em JPG e baixa um ZIP de evidências fora do repositório.

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = 'https://github.com/gvenancio12/yolo-edge-api.git'
WORKDIR = Path('/content/yolo-edge-api-aula6-yolo')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)
os.chdir(WORKDIR)
print('Diretório de trabalho:', Path.cwd())

In [ ]:
!pip -q install ultralytics

import torch
import ultralytics

print('Ultralytics:', ultralytics.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU não encontrada. No Colab: Runtime > Change runtime type > T4 GPU, reinicie e execute tudo novamente.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import yaml

DATASET_DIR = Path('datasets/epi-v1').resolve()
DATASET_YAML = DATASET_DIR / 'data.yaml'
assert DATASET_YAML.exists(), f'Dataset ausente: {DATASET_YAML}'
metadata = yaml.safe_load(DATASET_YAML.read_text())
print(DATASET_YAML.read_text())
for split in ('train', 'valid', 'test'):
    images = list((DATASET_DIR / split / 'images').glob('*'))
    labels = list((DATASET_DIR / split / 'labels').glob('*.txt'))
    print(f'{split}: {len(images)} imagens, {len(labels)} rótulos')
assert metadata['names'] == ['capacete', 'colete', 'pessoa']

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
RUNS_DIR = Path('runs/aula6-yolo').resolve()
model = YOLO('yolov8n.pt')
training = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=640,
    device=0,
    patience=0,
    project=str(RUNS_DIR),
    name='epi-yolov8n-50',
    exist_ok=True,
    plots=True,
)
SAVE_DIR = Path(training.save_dir)
BEST_PT = SAVE_DIR / 'weights' / 'best.pt'
assert BEST_PT.exists(), f'best.pt não encontrado: {BEST_PT}'
print('Treino concluído após', EPOCHS, 'épocas')
print('Pesos gerados:', BEST_PT)

In [ ]:
import json

detector = YOLO(str(BEST_PT))
metrics = detector.val(data=str(DATASET_YAML), split='val', device=0, plots=True)
final_metrics = {
    'mAP50': float(metrics.box.map50),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
}
print('Métricas finais:', json.dumps(final_metrics, indent=2))

In [ ]:
import cv2
from IPython.display import Image, display

sample_image = sorted((DATASET_DIR / 'test' / 'images').glob('*'))[0]
prediction = detector.predict(source=str(sample_image), conf=0.25, device=0, verbose=False)[0]
EXPORT_JPG = SAVE_DIR / 'deteccao_com_bounding_boxes.jpg'
assert cv2.imwrite(str(EXPORT_JPG), prediction.plot())
print('Imagem exportada:', EXPORT_JPG)
display(Image(filename=str(EXPORT_JPG)))

In [ ]:
import shutil

EVIDENCE_DIR = Path('/content/evidencias-aula6-yolo')
EVIDENCE_DIR.mkdir(exist_ok=True)
shutil.copy2(BEST_PT, EVIDENCE_DIR / 'best.pt')
shutil.copy2(DATASET_YAML, EVIDENCE_DIR / 'dataset.yaml')
shutil.copy2(EXPORT_JPG, EVIDENCE_DIR / EXPORT_JPG.name)
if (SAVE_DIR / 'results.csv').exists():
    shutil.copy2(SAVE_DIR / 'results.csv', EVIDENCE_DIR / 'results.csv')
(EVIDENCE_DIR / 'metricas_finais.json').write_text(json.dumps(final_metrics, indent=2))
archive = shutil.make_archive(str(EVIDENCE_DIR), 'zip', EVIDENCE_DIR)
print('Evidências fora do projeto:', EVIDENCE_DIR)
print('ZIP para download:', archive)

In [ ]:
from google.colab import files
files.download(archive)